# Step 2: Process Features

## Traitement des attributs

In [5]:
%load_ext autoreload
%autoreload 2


import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt

import os

from method_a_buffer import extract_buffer_feature
from feature_query import make_feature_query

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm
import rasterio
# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)


operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'


# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_pedestrian_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)

# Import des attributs
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info[attributs_info['include_in_index'] != False]


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loading pedestrian segments...


In [6]:
attributs_info

,Class,meta_attribute,attribute,include_in_index,attribute_source_path,initial_weight,class_weight,impact_attribut,file_name,geometry_type,method,how,value_column,buffer_size,filter_column,filter_values,crs,save_format
1,Agrément,bruit,bruit,True,bruit,0.4,0.4,defavorable,SPBR_SECTEUR_EXPOSE_AU_BRUIT_2025/SPBR_SECTEUR...,polygon,A,area_ratio,NaN,30,filtered,1,2056,"parquet, csv, gpkg"
2,Agrément,temperature,temperature,True,temperature,0.6,0.6,defavorable,CLIMAT_TEMPERATURE_14H00_P1_2020/CLIMAT_TEMPER...,point,A,raster,temperature,10,filtered,1,2056,"parquet, csv, gpkg"
3,Agrément,conflit_usage,conflit_usage,True,network_couche_OCT,0.5,0.5,defavorable,RP_final.shp,line,A,sum,Partage_us_score,1,filtered,1,2056,"parquet, csv, gpkg"
4,Agrément,vegetation,canopee,True,canopee,0.5,0.5,favorable,SIPV_ICA_MNC_2023-SHP/SIPV_ICA_MNC_2023.shp,polygon,A,length_area_ratio,NaN,30,filtered,1,2056,"parquet, csv, gpkg"
5,Attractivité,eau,eau,True,eau,0.3,0.3,favorable,LCE_GRAPHE_EAU-SHP/LCE_GRAPHE_EAU.shp,line,A,count,NaN,10,filtered,1,2056,"parquet, csv, gpkg"
6,Attractivité,espaces_ouverts,espaces_ouverts,True,espaces_ouverts,0.8,0.8,favorable,OBS_EQUIPEMENTS_ESPACES_PUB-SHP/OBS_EQUIPEMENT...,polygon,A,count,NaN,10,filtered,1,2056,"parquet, csv, gpkg"
7,Attractivité,proximite,rez_actif,True,rez_actif,0.5,0.5,favorable,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,point,A,count,NaN,10,filtered,1,2056,"parquet, csv, gpkg"
8,Attractivité,proximite,tp,True,tp,0.6,0.6,favorable,TPG_ARRETS-SHP/TPG_ARRETS.shp,point,A,count,NaN,200,filtered,1,2056,"parquet, csv, gpkg"
9,Attractivité,proximite,amenite,True,amenite,0.6,0.6,favorable,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,point,A,count,NaN,10,filtered,1,2056,"parquet, csv, gpkg"
10,Infrastructure,connectivite,connectivite,True,connectivite,0.7,0.7,favorable,NaN,line,A,sum,conn_branching_in_buffer,10,filtered,1,2056,"parquet, csv, gpkg"


In [7]:

# Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segmented_net
segmented_net = segmented_net[['geometry', 'segment_id', 'length']].copy()

'''# --- ADD THIS BLOCK ---
# Sample a subset of the network for faster testing
sample_fraction = 0.7  # 70% of all segments, adjust as needed
segmented_net = segmented_net.sample(frac=sample_fraction, random_state=42)

print(f"Testing on a sample of {len(segmented_net)} segments out of the total network.")
# ----------------------'''

print('Boucle sur chaque attribut... peut prendre du temps (30 mn)')

# Boucle sur le derniers attributs, first 3 only for testing
for _, row in attributs_info.iterrows():
    if row['include_in_index']:
        attribute_name = row['attribute']
        method = row['method']
        how = row['how']
        value_column = row['value_column']
        buffer_size = row['buffer_size']
        geometry_type = row['geometry_type']
        feature_query_expr = make_feature_query(
        row.get('filter_column'),
        row.get('filter_values')
    )

        # Charger la couche attribut depuis gpd_attributs
        print(f"Chargement de la couche: {attribute_name}")
        attribute_gdf = gpd.read_parquet(f"../../Data/output/step-2/parquet_attributs/{attribute_name}.parquet")
        attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

        # Appliquer la méthode
        
        if method == "A": # Buffer feature extraction
            print(f"Applying method {method} for attribute {attribute_name}...")
            attribute_df = extract_buffer_feature(
                segments_gdf = segmented_net,
                feature_gdf = attribute_gdf,
                feature_name=attribute_name,
                geom_kind=geometry_type,
                buffer_radius=buffer_size,
                how=how,
                value_column=value_column,
                crs_meter_epsg=operation_crs,
                feature_query  = feature_query_expr,
            )
            print(f"Buffer feature extracted for {attribute_name} with method {method}")
        
                # Debug duplicates
            if attribute_df[attribute_df.duplicated('segment_id')].shape[0] > 0:
                print("\nDEBUG: Found duplicate segment assignments")
                dupes = attribute_df[attribute_df.duplicated('segment_id', keep=False)]
                print(f"Number of segments with multiple zones: {len(dupes['segment_id'].unique())}")
            
                # Keep only the first occurrence for each segment_id
                attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')
                print("Dropped duplicates, keeping first occurrence")
            print(f"Spatial join computed for {attribute_name} with method {method}") 
        
        # Ajoute d'autres méthodes si besoin
        
        # Ajouter la colonne au GeoDataFrame principal
        segmented_net[f'{attribute_name}_{method}_{buffer_size}'] = attribute_df[attribute_name].fillna(0)
        print(f"Attribute {attribute_name} added to segmented_net.")

# Sauvegarder
print("Saving step 2 features to parquet...")
segmented_net.to_crs(target_crs).to_parquet(os.path.join(output_step2_path, "step2_features.parquet"), index=False)


Boucle sur chaque attribut... peut prendre du temps (30 mn)
Chargement de la couche: bruit
Applying method A for attribute bruit...
Buffer feature extracted for bruit with method A
Spatial join computed for bruit with method A
Attribute bruit added to segmented_net.
Chargement de la couche: temperature
Applying method A for attribute temperature...
Starting spatial join ...
Chek in joined is empty...
Joined is not empty.
Starting aggregation ...
Aggregation done.
Buffer feature extracted for temperature with method A
Spatial join computed for temperature with method A
Attribute temperature added to segmented_net.
Chargement de la couche: conflit_usage
Applying method A for attribute conflit_usage...
Buffer feature extracted for conflit_usage with method A
Spatial join computed for conflit_usage with method A
Attribute conflit_usage added to segmented_net.
Chargement de la couche: canopee
Applying method A for attribute canopee...
Buffer feature extracted for canopee with method A
Spati

In [8]:

segmented_net.head(20)


,geometry,segment_id,length,bruit_A_30,temperature_A_10,conflit_usage_A_1,canopee_A_30,eau_A_10,espaces_ouverts_A_10,rez_actif_A_10,tp_A_200,amenite_A_10,connectivite_A_10,largeur_trottoir_A_1,chemin_A_1,stationnement_genant_A_10,topographie_A_10,accident_A_10,zone_apaisee_A_10,zone_pietonne_A_10,vitesse_A_10
0,"LINESTRING (2505952.43 1117556.983, 2505957.52...",000000,18.611510,0.949764,32.117298,0.0,1.000000,0.0,0.0,0.0,3.0,0.0,8,6,0.0,0.0,3.1,0.0,0.000000,0.000000,1.000000
1,"LINESTRING (2504875.759 1116854.188, 2504891.4...",000001,17.373877,4.000000,33.267200,0.0,0.051274,0.0,1.0,0.0,5.0,0.0,88,4,1.0,0.0,5.6,1.0,0.000000,0.000000,0.942774
2,"LINESTRING (2500026.558 1117819.302, 2500041.9...",000002,50.000000,3.652566,33.157600,7.0,0.000000,0.0,4.0,2.0,9.0,2.0,378,24,1.0,6.0,62.9,8.0,0.000000,0.060493,0.939451
3,"LINESTRING (2500045.896 1117773.338, 2500047.9...",000003,7.198671,3.399097,0.000000,3.0,0.000000,0.0,2.0,0.0,8.0,0.0,237,9,0.0,1.0,21.9,5.0,0.000000,0.055699,0.944288
4,"LINESTRING (2498574.627 1115881.289, 2498530.0...",000004,45.125485,2.243178,32.843899,0.0,0.274643,0.0,2.0,1.0,4.0,1.0,215,28,0.0,0.0,49.6,0.0,0.000000,0.501680,0.498278
5,"LINESTRING (2503779.53 1116266.438, 2503790.87...",000005,11.749078,1.000000,31.430401,8.0,1.000000,0.0,0.0,0.0,7.0,0.0,25,5,0.0,0.0,12.2,1.0,0.000000,0.143946,1.286035
6,"LINESTRING (2496932.543 1119478.184, 2496913.2...",000006,22.405256,2.069667,0.000000,0.0,0.042198,0.0,1.0,0.0,2.0,0.0,10,14,0.0,0.0,27.3,0.0,0.652118,0.000000,0.347882
7,"LINESTRING (2504805.799 1117155.916, 2504813.8...",000007,12.111091,1.000000,29.096701,4.0,1.000000,0.0,0.0,0.0,0.0,0.0,1,0,0.0,0.0,10.2,1.0,1.000000,0.000000,0.000000
8,"LINESTRING (2498974.661 1122552.258, 2498967.7...",000008,18.010532,1.000000,0.000000,8.0,1.000000,0.0,0.0,0.0,4.0,0.0,48,3,0.0,0.0,49.3,1.0,0.000000,0.000000,1.523508
9,"LINESTRING (2499069.719 1114426.012, 2499064.2...",000009,14.050916,2.199339,32.784302,8.0,0.802692,0.0,2.0,0.0,8.0,0.0,54,7,0.0,0.0,36.7,2.0,0.000000,0.000000,1.000000


In [9]:
#import step2_features and convert to gpkg for QGIS use
segmented_net = gpd.read_parquet(os.path.join(output_step2_path, "step2_features.parquet"))
segmented_net.to_file(os.path.join(output_step2_path, "step2_features.gpkg"), driver="GPKG")
